# Diagnostic 1b: fixed random image embeddings (CLIP BASED)

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [2]:
import json

import numpy as np

from src import dataloader as data
from src import metrics
from src import train
from src.diagnostic_dataloader_random import load_multimodal_tensors_random_image
from src.models.classical_baseline import HORIZONS
from src.models.multimodal import MultimodalGRUPredictor

MODEL_NAME = 'CGM+time+clip (fixed random image, diagnostic 1)'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
ENCODER_NAME = 'clip'
IMG_FEAT_DIM = 768

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '4_diagnostic' / '1_diagnostic'
RESULTS_DIR = EXPERIMENT_DIR / 'results'
STABILITY_SUMMARY_PATH = (
    RESULTS_DIR / 'diagnostic1b_fixed_random_image_clip_validation_summary.json'
)


In [3]:
stats = data.load_cgm_stats()

_preview_loader = load_multimodal_tensors_random_image(
    'train', encoder_name=ENCODER_NAME, seed=SEEDS[0]
)
_sample_batch = next(iter(_preview_loader))

print('batch shapes:', [tuple(x.shape) for x in _sample_batch])
print('Diagnostic 1 uses train + validation only; the held-out test set is not loaded.')


batch shapes: [(256, 24), (256,), (256,), (256, 768), (256, 5)]
Diagnostic 1 uses train + validation only; the held-out test set is not loaded.


In [4]:
def forward_fn(model, batch, device):
    cgm, tsm, has_meal, img_feat, target = batch
    return (model(cgm.to(device), tsm.to(device), img_feat.to(device), has_meal.to(device)),
            target.to(device))

In [5]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []

for seed in SEEDS:
    print(f"Validation results — {MODEL_NAME} | seed={seed}\n")

    train_loader = load_multimodal_tensors_random_image(
        'train', encoder_name=ENCODER_NAME, seed=seed
    )
    val_loader = load_multimodal_tensors_random_image(
        'val', encoder_name=ENCODER_NAME, seed=seed, shuffle=False
    )

    train.set_seed(seed)
    model = MultimodalGRUPredictor(
        img_feat_dim=IMG_FEAT_DIM,
        gru_hidden_size=256,
        gru_num_layers=3,
        gru_dropout=0.2,
    )

    model, history = train.train_model(
        model,
        train_loader,
        val_loader,
        stats['cgm_mean'],
        stats['cgm_std'],
        lr=0.0001,
        weight_decay=1e-05,
        forward_fn=forward_fn,
        checkpoint_dir=(
            EXPERIMENT_DIR / 'checkpoints'
            / f'diagnostic1b_fixed_random_image_clip_seed{seed}'
        ),
        verbose=False,
        progress_bar=True,
        progress_desc=f'seed={seed}',
    )

    n_epochs = len(history['train_loss'])
    val_history = history.get('val_loss', [])
    best_epoch = int(np.argmin(val_history) + 1) if len(val_history) else n_epochs

    y_pred_val, y_true_val = train.predict(
        model, val_loader, forward_fn=forward_fn
    )
    val_results = metrics.evaluate(
        y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std']
    )

    seed_result = {
        'model': MODEL_NAME,
        'seed': seed,
        'n_epochs_trained': n_epochs,
        'best_epoch': best_epoch,
        'train_loss_history': [float(x) for x in history['train_loss']],
        'val_loss_history': [float(x) for x in val_history],
        'val_results': val_results,
    }
    all_results.append(seed_result)

    with open(
        RESULTS_DIR / f'diagnostic1b_fixed_random_image_clip_seed{seed}_results.json',
        'w',
    ) as f:
        json.dump(seed_result, f, indent=2)

    val_rmse = [round(val_results[h]['rmse'], 2) for h in HORIZONS]
    print(
        f"seed={seed:>3}  stopped at epoch {n_epochs:>3}/50  "
        f"best epoch={best_epoch:>3}  "
        f"validation RMSE @ {HORIZONS} min = {val_rmse}"
    )


Validation results — CGM+time+clip (fixed random image, diagnostic 1) | seed=7

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: MultimodalGRUPredictor  params: 1,326,293


seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

seed=  7  stopped at epoch  11/50  best epoch=  1  validation RMSE @ [15, 30, 60, 90, 120] min = [12.36, 18.12, 25.54, 30.39, 33.37]
Validation results — CGM+time+clip (fixed random image, diagnostic 1) | seed=14

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: MultimodalGRUPredictor  params: 1,326,293


seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

seed= 14  stopped at epoch  12/50  best epoch=  2  validation RMSE @ [15, 30, 60, 90, 120] min = [11.51, 17.72, 25.66, 30.76, 33.96]
Validation results — CGM+time+clip (fixed random image, diagnostic 1) | seed=21

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: MultimodalGRUPredictor  params: 1,326,293


...

seed= 56  stopped at epoch  11/50  best epoch=  1  validation RMSE @ [15, 30, 60, 90, 120] min = [12.49, 18.17, 25.62, 30.45, 33.46]
Validation results — CGM+time+clip (fixed random image, diagnostic 1) | seed=63

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: MultimodalGRUPredictor  params: 1,326,293


seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

seed= 63  stopped at epoch  11/50  best epoch=  1  validation RMSE @ [15, 30, 60, 90, 120] min = [12.36, 18.05, 25.53, 30.39, 33.41]
Validation results — CGM+time+clip (fixed random image, diagnostic 1) | seed=70

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: MultimodalGRUPredictor  params: 1,326,293


seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

seed= 70  stopped at epoch  11/50  best epoch=  1  validation RMSE @ [15, 30, 60, 90, 120] min = [12.21, 17.98, 25.49, 30.32, 33.34]


In [6]:
# Diagnostic summary: validation only.
# The held-out test set remains untouched during this model-development diagnostic.

METRICS = ['rmse', 'mae', 'grmse']
summary = {}

for h in HORIZONS:
    summary[h] = {}
    for metric_name in METRICS:
        values = [r['val_results'][h][metric_name] for r in all_results]
        summary[h][metric_name] = {
            'mean': float(np.mean(values)),
            'sd_across_seeds': float(np.std(values, ddof=1)),
        }

best_epochs = [r['best_epoch'] for r in all_results]
stopping_epochs = [r['n_epochs_trained'] for r in all_results]

final_loss_gaps = []
for r in all_results:
    if r['val_loss_history']:
        final_loss_gaps.append(
            float(r['val_loss_history'][-1] - r['train_loss_history'][-1])
        )

print(f'=== {MODEL_NAME}: validation summary across seeds ===')
for h in HORIZONS:
    s = summary[h]
    print(
        f"{h:>3}-min: "
        f"RMSE = {s['rmse']['mean']:.2f} +/- {s['rmse']['sd_across_seeds']:.2f}   "
        f"MAE = {s['mae']['mean']:.2f} +/- {s['mae']['sd_across_seeds']:.2f}   "
        f"gRMSE = {s['grmse']['mean']:.2f} +/- {s['grmse']['sd_across_seeds']:.2f}"
    )

print(
    f"Best epoch: {np.mean(best_epochs):.2f} +/- {np.std(best_epochs, ddof=1):.2f}; "
    f"stopping epoch: {np.mean(stopping_epochs):.2f} +/- "
    f"{np.std(stopping_epochs, ddof=1):.2f}"
)

if final_loss_gaps:
    print(
        f"Final normalized val-train loss gap: "
        f"{np.mean(final_loss_gaps):.6f} +/- {np.std(final_loss_gaps, ddof=1):.6f}"
    )

with open(STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump(
        {
            'model': MODEL_NAME,
            'seeds': SEEDS,
            'n_seeds': len(SEEDS),
            'per_horizon_validation_summary': summary,
            'best_epochs': best_epochs,
            'stopping_epochs': stopping_epochs,
            'final_normalized_val_minus_train_loss_gaps': final_loss_gaps,
            'test_set_evaluated': False,
        },
        f,
        indent=2,
    )

print(f'\nsaved per-seed validation results to {RESULTS_DIR}')
print(f'saved validation summary to {STABILITY_SUMMARY_PATH}')


=== CGM+time+clip (fixed random image, diagnostic 1): validation summary across seeds ===
 15-min: RMSE = 12.28 +/- 0.29   MAE = 8.40 +/- 0.25   gRMSE = 13.48 +/- 0.33
 30-min: RMSE = 18.05 +/- 0.15   MAE = 12.15 +/- 0.17   gRMSE = 20.50 +/- 0.25
 60-min: RMSE = 25.56 +/- 0.08   MAE = 16.93 +/- 0.20   gRMSE = 30.43 +/- 0.26
 90-min: RMSE = 30.44 +/- 0.13   MAE = 20.28 +/- 0.24   gRMSE = 37.10 +/- 0.32
120-min: RMSE = 33.49 +/- 0.19   MAE = 22.82 +/- 0.18   gRMSE = 41.10 +/- 0.36
Best epoch: 1.10 +/- 0.32; stopping epoch: 11.10 +/- 0.32
Final normalized val-train loss gap: 0.139791 +/- 0.004627

saved per-seed validation results to /content/drive/MyDrive/dissertation/experiments/4_diagnostic/1_diagnostic/results
saved validation summary to /content/drive/MyDrive/dissertation/experiments/4_diagnostic/1_diagnostic/results/diagnostic1b_fixed_random_image_clip_validation_summary.json


In [17]:
from scipy import stats
import numpy as np
import json

CORRECT_RESULTS_DIR = (
    PROJECT_ROOT
    / 'experiments'
    / '3_initial_multimodal'
    / 'multimodal'
    / 'results'
    / 'gated_clip'
)

correct_results = []

for seed in SEEDS:
    path = CORRECT_RESULTS_DIR / f'gated_cgm_time_clip_seed{seed}_results.json'

    with open(path, 'r') as f:
        correct_results.append(json.load(f))

random_results = all_results


print('=== Paired comparison: correct image vs. fixed random image ===')
print(
    f"{'horizon':>10}"
    f"{'correct':>12}"
    f"{'random':>12}"
    f"{'diff':>10}"
    f"{'95% CI':>22}"
    f"{'p-value':>12}"
)

paired_results = {}

for h in HORIZONS:

    correct = np.array([
        r['val_results'][str(h)]['rmse']
        for r in correct_results
    ])

    random = np.array([
        r['val_results'][h]['rmse']
        for r in random_results
    ])

    diff = correct - random

    mean_diff = np.mean(diff)
    sd_diff = np.std(diff, ddof=1)
    se_diff = stats.sem(diff)

    ci_low, ci_high = stats.t.interval(
        0.95,
        df=len(diff) - 1,
        loc=mean_diff,
        scale=se_diff,
    )

    t_stat, p_value = stats.ttest_rel(correct, random)

    paired_results[h] = {
        'correct_mean_rmse': float(np.mean(correct)),
        'random_mean_rmse': float(np.mean(random)),
        'mean_difference_correct_minus_random': float(mean_diff),
        'sd_difference': float(sd_diff),
        'ci95': [float(ci_low), float(ci_high)],
        't_statistic': float(t_stat),
        'p_value': float(p_value),
    }

    print(
        f"{h:>8}min"
        f"{np.mean(correct):>12.2f}"
        f"{np.mean(random):>12.2f}"
        f"{mean_diff:>+10.2f}"
        f"{f'[{ci_low:.2f}, {ci_high:.2f}]':>22}"
        f"{p_value:>12.4f}"
    )

=== Paired comparison: correct image vs. fixed random image ===
   horizon     correct      random      diff                95% CI     p-value
      15min       12.23       12.28     -0.05         [-0.30, 0.20]      0.6580
      30min       18.10       18.05     +0.05         [-0.04, 0.14]      0.2110
      60min       25.75       25.56     +0.19          [0.09, 0.29]      0.0020
      90min       30.56       30.44     +0.12          [0.02, 0.21]      0.0191
     120min       33.33       33.49     -0.15        [-0.30, -0.01]      0.0368
